In [5]:
import sys
print(sys.executable)
print(sys.path[:2])

c:\Users\USUARIO\Documents\Dev\finance\finance-backend\.venv\Scripts\python.exe
['C:\\Users\\USUARIO\\AppData\\Local\\Programs\\Python312', 'C:\\Users\\USUARIO\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip']


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from app.core import config, extract, processing
from pathlib import Path
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
from app.core.utils import get_exchange_rate, convert_currency
import json

In [2]:
df = extract.extract_dataset(config.DATASET_ROOT_PATH, 2025)
df

,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO,MONEDA,ENTIDAD
0,2025-01-01 00:00:00,ABONO INTERESES AHORROS,34.88,25600347.67,COP,BC
1,2025-01-01 00:00:00,TRANSFERENCIA A NEQUI,-136850.00,25463497.67,COP,BC
2,2025-01-02 00:00:00,COMPRA EN IGSA HL S,-92324.00,25371173.67,COP,BC
3,2025-01-02 00:00:00,COMPRA INTL PAYPAL DISNEYPLU,-55423.01,25315750.66,COP,BC
4,2025-01-02 00:00:00,PAGO AUTOM TC MASTER PESOS,-35200.00,25280550.66,COP,BC
...,...,...,...,...,...,...
996,2025-10-13 09:29:59,Card charge (ESPRIT),-15.92,767.42,USD,PAYONEER
997,2025-10-13 09:29:59,Card charge (PATPRIMO 227 IBAGUE MU),-45.12,691.48,USD,PAYONEER
998,2025-10-13 09:29:59,Card charge (SURTIPLAZA ACQUA),-30.82,736.60,USD,PAYONEER
999,2025-11-10 15:16:41,Payment from Jala University,588.00,1226.37,USD,PAYONEER


In [4]:
df

,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO,MONEDA,ENTIDAD
0,2025-01-01 00:00:00,ABONO INTERESES AHORROS,34.88,25600347.67,COP,BC
1,2025-01-01 00:00:00,TRANSFERENCIA A NEQUI,-136850.00,25463497.67,COP,BC
2,2025-01-02 00:00:00,COMPRA EN IGSA HL S,-92324.00,25371173.67,COP,BC
3,2025-01-02 00:00:00,COMPRA INTL PAYPAL DISNEYPLU,-55423.01,25315750.66,COP,BC
4,2025-01-02 00:00:00,PAGO AUTOM TC MASTER PESOS,-35200.00,25280550.66,COP,BC
...,...,...,...,...,...,...
996,2025-10-13 09:29:59,Card charge (ESPRIT),-15.92,767.42,USD,PAYONEER
997,2025-10-13 09:29:59,Card charge (PATPRIMO 227 IBAGUE MU),-45.12,691.48,USD,PAYONEER
998,2025-10-13 09:29:59,Card charge (SURTIPLAZA ACQUA),-30.82,736.60,USD,PAYONEER
999,2025-11-10 15:16:41,Payment from Jala University,588.00,1226.37,USD,PAYONEER


In [50]:
def summary_report(df, currency='USD'):
    df = df.copy()

    df['FECHA'] = pd.to_datetime(df['FECHA'])

    rates = {
        m: get_exchange_rate(m)
        for m in df['MONEDA'].unique()
        if m != currency
    }
   
    print(df[['CREDIT/DEBIT', 'SALDO']].dtypes)
    for m, rate in rates.items():
        mask = df['MONEDA'] == m
        print(m, rate, mask.sum())
        df.loc[mask, 'CREDIT/DEBIT'].head()
        for col in ['CREDIT/DEBIT', 'SALDO']:
            df.loc[mask, col] = df.loc[mask, col] * rate

    
    df['MONEDA'] = currency
    df['INGRESO'] = df['CREDIT/DEBIT'].clip(lower=0)
    df['EGRESO']  = (-df['CREDIT/DEBIT']).clip(lower=0)

    

    return df
    # resumen = (
    #     df
    #     .groupby(['MONEDA', 'ENTIDAD'], as_index=False)
    #     .agg(
    #         INGRESO=('INGRESO', 'sum'),
    #         EGRESO=('EGRESO', 'sum')
    #     )
    # )

    # saldo = (
    #     df.loc[
    #         df.groupby(['MONEDA', 'ENTIDAD'])['FECHA'].idxmax(),
    #         ['MONEDA', 'ENTIDAD', 'SALDO']
    #     ]
    # )

    # resumen = resumen.merge(saldo, on=['MONEDA', 'ENTIDAD'], how='left')
    

    
    # resumen.loc[:, 'MONEDA'] = currency
    # resumen['BALANCE'] = resumen['INGRESO'] - resumen['EGRESO']
    # # --- Resumen global ---
    # total = {
    #     "currency": currency,
    #     "income": round(resumen['INGRESO'].sum(), 2),
    #     "expenses": round(resumen['EGRESO'].sum(), 2),
    #     "totalBalance": round(resumen['SALDO'].sum(), 2),
    #      "entities": resumen['ENTIDAD'].nunique()
    #  }
    # return total



In [51]:
summary_report(df, currency='USD')

CREDIT/DEBIT    float64
SALDO           float64
dtype: object
COP 1 946


,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO,MONEDA,ENTIDAD,INGRESO,EGRESO
0,2025-01-01 00:00:00,ABONO INTERESES AHORROS,34.88,25600347.67,USD,BC,34.88,0.00
1,2025-01-01 00:00:00,TRANSFERENCIA A NEQUI,-136850.00,25463497.67,USD,BC,0.00,136850.00
2,2025-01-02 00:00:00,COMPRA EN IGSA HL S,-92324.00,25371173.67,USD,BC,0.00,92324.00
3,2025-01-02 00:00:00,COMPRA INTL PAYPAL DISNEYPLU,-55423.01,25315750.66,USD,BC,0.00,55423.01
4,2025-01-02 00:00:00,PAGO AUTOM TC MASTER PESOS,-35200.00,25280550.66,USD,BC,0.00,35200.00
...,...,...,...,...,...,...,...,...
996,2025-10-13 09:29:59,Card charge (ESPRIT),-15.92,767.42,USD,PAYONEER,0.00,15.92
997,2025-10-13 09:29:59,Card charge (PATPRIMO 227 IBAGUE MU),-45.12,691.48,USD,PAYONEER,0.00,45.12
998,2025-10-13 09:29:59,Card charge (SURTIPLAZA ACQUA),-30.82,736.60,USD,PAYONEER,0.00,30.82
999,2025-11-10 15:16:41,Payment from Jala University,588.00,1226.37,USD,PAYONEER,588.00,0.00


In [3]:
resumen = processing.summary_report(df,currency='USD')
resumen

{'currency': 'USD',
 'income': np.float64(120697343.44),
 'expenses': np.float64(112064553.57),
 'totalBalance': np.float64(34286746.62),
 'entities': 3}

In [4]:
def summary_report(df):
    df['FECHA'] = pd.to_datetime(df['FECHA'])
    df['MES'] = df['FECHA'].dt.to_period('M').astype(str)

    df['INGRESO'] = df['CREDIT/DEBIT'].where(df['CREDIT/DEBIT'] > 0, 0)
    df['EGRESO']  = df['CREDIT/DEBIT'].where(df['CREDIT/DEBIT'] < 0, 0).abs()
    resumen = (
        df
        .groupby(['MONEDA','ENTIDAD'], as_index=False)
        .agg({
            'INGRESO': 'sum',
            'EGRESO': 'sum'
        })
    )
    saldo = (
        df
        .sort_values('FECHA')
        .groupby(['MONEDA', 'ENTIDAD'], as_index=False)
        .last()
        [['MONEDA', 'ENTIDAD', 'SALDO']]
    )
    resumen = resumen.merge(
        saldo,
        on=['MONEDA', 'ENTIDAD'],
        how='left'
    )

    resumen['BALANCE'] = resumen['INGRESO'] - resumen['EGRESO']
    resumen['INGRESO'] = resumen['INGRESO'].map('{:,.0f}'.format)
    resumen['EGRESO']  = resumen['EGRESO'].map('{:,.0f}'.format)
    resumen['SALDO']   = resumen['SALDO'].map('{:,.0f}'.format)
    resumen['BALANCE'] = resumen['BALANCE'].map('{:,.0f}'.format)
    return resumen

In [ ]:
def summary_report(df):
    df = df.copy()

    df['FECHA'] = pd.to_datetime(df['FECHA'])

    df['INGRESO'] = df['CREDIT/DEBIT'].clip(lower=0)
    df['EGRESO']  = (-df['CREDIT/DEBIT']).clip(lower=0)

    resumen = (
        df
        .groupby(['MONEDA', 'ENTIDAD'], as_index=False)
        .agg(
            INGRESO=('INGRESO', 'sum'),
            EGRESO=('EGRESO', 'sum')
        )
    )

    saldo = (
        df.loc[
            df.groupby(['MONEDA', 'ENTIDAD'])['FECHA'].idxmax(),
            ['MONEDA', 'ENTIDAD', 'SALDO']
        ]
    )

    resumen = resumen.merge(saldo, on=['MONEDA', 'ENTIDAD'], how='left')
    resumen['BALANCE'] = resumen['INGRESO'] - resumen['EGRESO']

    return resumen


In [58]:
report = summary_report(df)

In [59]:
report

,MONEDA,ENTIDAD,INGRESO,EGRESO,SALDO,BALANCE
0,COP,BC,"120,692,492","112,063,895","34,260,909","8,628,596"
1,USD,AMERANT,"2,500",121,"24,023","2,379"
2,USD,PAYONEER,"2,352",538,"1,814","1,814"


In [10]:


resumen = (
    df
    .groupby(['ENTIDAD', 'MONEDA'])['CREDIT/DEBIT']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

resumen

MONEDA,ENTIDAD,COP,USD
0,AMERANT,0.00,2379.07
1,BC,8628596.43,0.00
2,PAYONEER,0.00,1814.37


In [2]:
df = extract.extractExtractosFromFolderYearBC(config.DATASET_ROOT_PATH, 2024)
df

,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO,MONEDA,ENTIDAD
0,2024-01-01,ABONO INTERESES AHORROS,8.63,6452567.86,COP,BC
1,2024-01-01,TRANSFERENCIA CTA SUC VIRTUAL,-39450.00,6413117.86,COP,BC
2,2024-01-01,TRANSFERENCIA CTA SUC VIRTUAL,-110362.00,6302755.86,COP,BC
3,2024-01-02,ABONO INTERESES AHORROS,8.58,6302764.44,COP,BC
4,2024-01-02,COMPRA INTL PAYPAL DISNEYPLU,-32410.98,6270353.46,COP,BC
...,...,...,...,...,...,...
915,2024-12-31,ABONO INTERESES AHORROS,35.06,25822255.79,COP,BC
916,2024-12-31,PAGO IBAL S.A. ESP,-39800.00,25782455.79,COP,BC
917,2024-12-31,COMPRA EN SUPERMERCA,-84900.00,25697555.79,COP,BC
918,2024-12-31,COMPRA EN SUPERMERCA,-25243.00,25672312.79,COP,BC


In [3]:
df_2025 = extract.extract_dataset(config.DATASET_ROOT_PATH, year=2024)
df_2025

,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO,MONEDA,ENTIDAD
0,2024-01-01 00:00:00,ABONO INTERESES AHORROS,8.63,6452567.86,COP,BC
1,2024-01-01 00:00:00,TRANSFERENCIA CTA SUC VIRTUAL,-39450.00,6413117.86,COP,BC
2,2024-01-01 00:00:00,TRANSFERENCIA CTA SUC VIRTUAL,-110362.00,6302755.86,COP,BC
3,2024-01-02 00:00:00,ABONO INTERESES AHORROS,8.58,6302764.44,COP,BC
4,2024-01-02 00:00:00,COMPRA INTL PAYPAL DISNEYPLU,-32410.98,6270353.46,COP,BC
...,...,...,...,...,...,...
1069,2024-10-31,INTEREST CREDIT,0.18,21632.02,USD,AMERANT
1070,2024-11-30,FED TAX WITHHELD,-0.04,21631.98,USD,AMERANT
1071,2024-11-30,INTEREST CREDIT,0.18,21632.16,USD,AMERANT
1072,2024-12-31,FED TAX WITHHELD,-0.04,21632.12,USD,AMERANT


In [4]:
df_period= extract.extract_from_period_xls(config.DATASET_ROOT_PATH +"/2025/BC/Periodos/MovimientosTusCuentasBancolombia02Ene2026.xlsx")
df_period


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\USUARIO\\Documents\\Dev\\finance\\dataset/2025/BC/Periodos/MovimientosTusCuentasBancolombia02Ene2026.xlsx'

In [7]:
df_period.rename(columns={'Fecha':'FECHA','Valor':'CREDIT/DEBIT','Descripción':'DESCRIPCION'}, inplace=True)
df_period.drop(columns=['Referencia'], inplace=True)
df_period['FECHA'] = pd.to_datetime(df_period['FECHA'], format='%d/%m/%Y')
df_period['FECHA'].set_index = True
df_period.sort_index(ascending=False, inplace=True)

KeyError: "['Referencia'] not found in axis"

In [14]:
df_period

,FECHA,DESCRIPCION,CREDIT/DEBIT
231,2025-10-03 05:00:00,PAGO PSE Wingo Aerorepublica,-2606588.00
230,2025-10-03 05:00:00,ABONO INTERESES AHORROS,78.34
229,2025-10-04 05:00:00,COMPRA EN LEVIS ACQU,-319900.00
228,2025-10-04 05:00:00,ABONO INTERESES AHORROS,35.60
227,2025-10-05 05:00:00,PAGO PSE COMUNICACION CELULAR,-32000.00
...,...,...,...
4,2025-12-30 05:00:00,TRANSFERENCIA CTA SUC VIRTUAL,-14700.00
3,2025-12-30 05:00:00,ABONO INTERESES AHORROS,48.24
2,2025-12-31 05:00:00,ABONO INTERESES AHORROS,46.93
1,2025-12-31 05:00:00,PAGO PSE COMUNICACION CELULAR,-32000.00


In [ ]:
df = extract.extractExtractosFromFolderYearBC(config.DATASET_ROOT_PATH, 2025)
df

Extracting data from folder: C:\Users\USUARIO\Documents\Dev\finance\dataset/2025/BC/Extractos
Processing file (<function extractExtractosFromFolderYearBC at 0x0000022CB3D16C00>): Extracto_202503_Cuentas_de ahorro_T1.xlsx
Processing file (<function extractExtractosFromFolderYearBC at 0x0000022CB3D16C00>): Extracto_202506_Cuentas_de ahorro_T2.xlsx
Processing file (<function extractExtractosFromFolderYearBC at 0x0000022CB3D16C00>): Extracto_202509_Cuentas_de ahorro_T3.xlsx
Processing file (<function extractExtractosFromFolderYearBC at 0x0000022CB3D16C00>): MovimientosTusCuentasBancolombia02Ene2026.xlsx
Processing file (<function extract_from_period_xls at 0x0000022CB3D16A20>): C:\Users\USUARIO\Documents\Dev\finance\dataset/2025/BC/Periodos\MovimientosTusCuentasBancolombia02Ene2026.xlsx
                  FECHA                    DESCRIPCION  CREDIT/DEBIT
231 2025-10-03 05:00:00   PAGO PSE Wingo Aerorepublica   -2606588.00
230 2025-10-03 05:00:00        ABONO INTERESES AHORROS         78.34

TypeError: unsupported operand type(s) for +: 'Timestamp' and 'str'

In [5]:
# Cargar DataFrame
year = 2024
try:
    
    df = extract.extract_dataset(config.DATASET_ROOT_PATH, year)
    
except Exception as e:
    print(f"Error al cargar el dataset para el año {year}: {e}")
        # Resumen en la moneda deseada
df

Extracting data from folder: C:\Users\USUARIO\Documents\Dev\finance\dataset/2024/BC/Extractos
Error extracting BC data for year 2024: exceptions must derive from BaseException


,FECHA,DESCRIPCION,SALDO,CREDIT/DEBIT,MONEDA,ENTIDAD
0,2024-01-02,POS PURCHASE MERCHANT PURCHASE TERMINAL 755483...,18354.26,-23.65,USD,AMERANT
1,2024-01-02,EFT SERVICE CHARGE SVC CHG INTRNTL TR AN,18286.48,-0.96,USD,AMERANT
2,2024-01-02,EFT SERVICE CHARGE SVC CHG INTRNTL TR AN,18287.44,-0.88,USD,AMERANT
3,2024-01-02,EFT SERVICE CHARGE SVC CHG INTRNTL TR AN,18288.32,-0.71,USD,AMERANT
4,2024-01-02,EFT SERVICE CHARGE SVC CHG INTRNTL TR AN,18289.03,-0.45,USD,AMERANT
...,...,...,...,...,...,...
149,2024-10-31,INTEREST CREDIT,21632.02,0.18,USD,AMERANT
150,2024-11-30,FED TAX WITHHELD,21631.98,-0.04,USD,AMERANT
151,2024-11-30,INTEREST CREDIT,21632.16,0.18,USD,AMERANT
152,2024-12-31,FED TAX WITHHELD,21632.12,-0.04,USD,AMERANT


In [4]:
try:
    df = extract.get_balanceMonthly(df, usd_to_cop=3700, currency='USD')
except Exception as e:
    print(f"Error al convertir moneda: {e}")

       FECHA                     DESCRIPCION  CREDIT/DEBIT       SALDO MONEDA  \
0 2024-01-01         ABONO INTERESES AHORROS          8.63  6452567.86    COP   
1 2024-01-01   TRANSFERENCIA CTA SUC VIRTUAL     -39450.00  6413117.86    COP   
2 2024-01-01   TRANSFERENCIA CTA SUC VIRTUAL    -110362.00  6302755.86    COP   
3 2024-01-02         ABONO INTERESES AHORROS          8.58  6302764.44    COP   
4 2024-01-02  COMPRA INTL  PAYPAL  DISNEYPLU     -32410.98  6270353.46    COP   

  ENTIDAD      MES  
0      BC  2024-01  
1      BC  2024-01  
2      BC  2024-01  
3      BC  2024-01  
4      BC  2024-01  
       FECHA                     DESCRIPCION  CREDIT/DEBIT       SALDO MONEDA  \
0 2024-01-01         ABONO INTERESES AHORROS          8.63  6452567.86    COP   
1 2024-01-01   TRANSFERENCIA CTA SUC VIRTUAL     -39450.00  6413117.86    COP   
2 2024-01-01   TRANSFERENCIA CTA SUC VIRTUAL    -110362.00  6302755.86    COP   
3 2024-01-02         ABONO INTERESES AHORROS          8.58  630

In [3]:
try:
    json_balance = extract.getBalanceByEntity(
        df, usd_to_cop=3700, currency='USD'
    )
except Exception as e:
    print(f"Error al procesar el dataset para el año {year}: {e}")  

      ENTIDAD        SALDO MONEDA  BALANCE_FINAL
153   AMERANT     21632.30    USD        21632.3
1073       BC  25600312.79    COP         6919.0
{'currency': 'USD', 'entities': [{'ENTIDAD': 'AMERANT', 'BALANCE_FINAL': 21632.3}, {'ENTIDAD': 'BC', 'BALANCE_FINAL': 6919.0}]}


In [4]:
json_balance

[{'ENTIDAD': 'AMERANT', 'BALANCE_FINAL': 21632.3},
 {'ENTIDAD': 'BC', 'BALANCE_FINAL': 6919.0}]

In [4]:

try:
    
    resumen = processing.summarize_dataframe(df, usd_to_cop=3700, currency='USD')
    print(resumen)
except Exception as e:
    print(f"Error al procesar el dataset para el año {year}: {e}")  

{'totalBalance': np.float64(3254.39), 'entities': 1, 'income': np.float64(5960.46), 'expenses': np.float64(2706.07), 'currency': 'USD'}


In [3]:
r = extract.getBalanceByMonth(df, usd_to_cop=3700, currency='USD')
r

[{'MES': '2024-01',
  'total_balance': 2194.99,
  'income': 1843.79,
  'expenses': 2382.0399999999995},
 {'MES': '2024-02',
  'total_balance': 1283.38,
  'income': 2240.99,
  'expenses': 3154.84},
 {'MES': '2024-03',
  'total_balance': 928.6,
  'income': 2129.73,
  'expenses': 2582.12},
 {'MES': '2024-04',
  'total_balance': 2039.12,
  'income': 2677.3399999999997,
  'expenses': 2003.1399999999999},
 {'MES': '2024-05',
  'total_balance': 1689.97,
  'income': 2129.52,
  'expenses': 2006.31},
 {'MES': '2024-06',
  'total_balance': 1496.36,
  'income': 3100.7300000000005,
  'expenses': 3448.27},
 {'MES': '2024-07',
  'total_balance': 1883.55,
  'income': 5691.820000000001,
  'expenses': 1855.3000000000002},
 {'MES': '2024-08',
  'total_balance': 1385.65,
  'income': 3597.24,
  'expenses': 2769.7400000000002},
 {'MES': '2024-09',
  'total_balance': 21249.2,
  'income': 2594.6800000000003,
  'expenses': 2455.67},
 {'MES': '2024-10',
  'total_balance': 21631.84,
  'income': 2706.9,
  'expens

## Año completo evaluando los 4 extratos trimestrales

- Se extraen los extractos
- Se pasa el folder root dataset/
- y el año dataset/YEAR
- En ese folder se encuentran todos los extratos del año (4) o los que hayan si se lee a mediados de año, al menos despues del primer trimestre (abril)
- se leen y se dejan en un df con los valores ['FECHA', 'DESCRIPCION', 'CREDIT/DEBIT', 'SALDO']

In [2]:
config.DATASET_ROOT_PATH

'c:\\Users\\USUARIO\\Documents\\Dev\\finance\\dataset'

In [2]:
df_amerant = extract.extract_amerant(config.DATASET_ROOT_PATH,2025)
df_amerant

,FECHA,DESCRIPCION,SALDO,CREDIT/DEBIT,MONEDA
0,2025-01-16,SERVICE CHARGE INCOMING WIRE FEE,21632.16,-12.00,USD
1,2025-01-31,FED TAX WITHHELD,21632.12,-0.04,USD
2,2025-01-31,INTEREST CREDIT,21632.30,0.18,USD
3,2025-02-12,CREDIT MEMO FEE REIMBURSEMENT,21633.95,1.65,USD
4,2025-02-28,FED TAX WITHHELD,21633.91,-0.04,USD
5,2025-02-28,INTEREST CREDIT,21634.08,0.17,USD
6,2025-03-12,WIRE IN-DOM c0a895e7-3467-483c -992b-502d4091b...,21845.58,211.50,USD
7,2025-03-12,SERVICE CHARGE INCOMING WIRE FEE,21833.58,-12.00,USD
8,2025-03-31,FED TAX WITHHELD,21833.54,-0.04,USD
9,2025-03-31,INTEREST CREDIT,21833.72,0.18,USD


In [ ]:
def get_subfolders(root_folder: str) -> list[str]:
    """
    Retorna todos los subdirectorios a partir de un folder raíz.
    """
    root = Path(root_folder)

    if not root.exists() or not root.is_dir():
        raise ValueError(f"Ruta inválida: {root_folder}")

    return [str(p) for p in root.rglob('*') if p.is_dir()]

get_subfolders(config.DATASET_ROOT_PATH)

In [6]:
df_2025 = extract.extractFromFolderYear(config.DATASET_ROOT_PATH, 2025)
df_2025

Extracting data from folder: c:\Users\USUARIO\Documents\Dev\finance\dataset/BC/2025


,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO
0,2025-01-01,ABONO INTERESES AHORROS,34.88,25600347.67
1,2025-01-01,TRANSFERENCIA A NEQUI,-136850.00,25463497.67
2,2025-01-02,COMPRA EN IGSA HL S,-92324.00,25371173.67
3,2025-01-02,COMPRA INTL PAYPAL DISNEYPLU,-55423.01,25315750.66
4,2025-01-02,PAGO AUTOM TC MASTER PESOS,-35200.00,25280550.66
...,...,...,...,...
705,2025-09-30,RETIRO CAJERO UNIVERSIDAD IBA,-1000000.00,29212565.80
706,2025-09-30,RETIRO CAJERO UNIVERSIDAD IBA,-500000.00,28712565.80
707,2025-09-30,TRASLADO VIRTUAL OTROS BANCOS,-240245.00,28472320.80
708,2025-09-30,TRANSFERENCIA CTA SUC VIRTUAL,-120000.00,28352320.80


In [5]:
df = extract.extractExtractosFromFolderYearBC(config.DATASET_ROOT_PATH, 2024)
df

TypeError: exceptions must derive from BaseException

In [4]:
df = extract.extract_payoneer(config.DATASET_ROOT_PATH, 2025)
df

,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO,MONEDA,ENTIDAD
0,2025-12-23 15:16:41,Payment from Jala University,588.00,1814.37,USD,PAYONEER
1,2025-11-10 15:16:41,Payment from Jala University,588.00,1226.37,USD,PAYONEER
2,2025-10-13 09:29:59,Card charge (SURTIPLAZA ACQUA),-30.82,736.60,USD,PAYONEER
3,2025-10-13 09:29:59,Card charge (PATPRIMO 227 IBAGUE MU),-45.12,691.48,USD,PAYONEER
4,2025-10-13 09:29:59,Card charge (ESPRIT),-15.92,767.42,USD,PAYONEER
5,2025-10-13 09:29:59,Card charge (FALABELLA TDA DPTO ACQ),-53.11,638.37,USD,PAYONEER
6,2025-10-12 09:34:39,Card charge (BOLD*MIS CARNES PARR),-28.94,783.34,USD,PAYONEER
7,2025-10-12 08:10:12,Card charge (RESTAURANTE DONDE CHUC),-44.24,812.28,USD,PAYONEER
8,2025-10-05 09:09:06,Card charge (SUPERMERCADO POPULAR 1),-9.60,856.52,USD,PAYONEER
9,2025-10-05 09:04:46,Card charge (FALABELLA TDA DPTO ACQ),-82.32,866.12,USD,PAYONEER


In [6]:
config.DATASET_ROOT_PATH

'c:\\Users\\USUARIO\\Documents\\Dev\\finance\\dataset'

In [5]:


ent = extract.extract_dataset(config.DATASET_ROOT_PATH, 2025)
ent

,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO,MONEDA,ENTIDAD
0,2025-01-01 00:00:00,ABONO INTERESES AHORROS,34.88,25600347.67,COP,BC
1,2025-01-01 00:00:00,TRANSFERENCIA A NEQUI,-136850.00,25463497.67,COP,BC
2,2025-01-02 00:00:00,COMPRA EN IGSA HL S,-92324.00,25371173.67,COP,BC
3,2025-01-02 00:00:00,COMPRA INTL PAYPAL DISNEYPLU,-55423.01,25315750.66,COP,BC
4,2025-01-02 00:00:00,PAGO AUTOM TC MASTER PESOS,-35200.00,25280550.66,COP,BC
...,...,...,...,...,...,...
760,2025-10-13 09:29:59,Card charge (ESPRIT),-15.92,767.42,USD,PAYONEER
761,2025-10-13 09:29:59,Card charge (PATPRIMO 227 IBAGUE MU),-45.12,691.48,USD,PAYONEER
762,2025-10-13 09:29:59,Card charge (SURTIPLAZA ACQUA),-30.82,736.60,USD,PAYONEER
763,2025-11-10 15:16:41,Payment from Jala University,588.00,1226.37,USD,PAYONEER


In [4]:
config.DATASET_ROOT_PATH

'C:\\Users\\USUARIO\\Documents\\Dev\\finance\\dataset'

In [9]:
# Cargar DataFrame
year = 2025
df = extract.extract_dataset(config.DATASET_ROOT_PATH, year)
df

,FECHA,DESCRIPCION,CREDIT/DEBIT,SALDO,MONEDA,ENTIDAD
0,2025-01-01 00:00:00,ABONO INTERESES AHORROS,34.88,25600347.67,COP,BC
1,2025-01-01 00:00:00,TRANSFERENCIA A NEQUI,-136850.00,25463497.67,COP,BC
2,2025-01-02 00:00:00,COMPRA EN IGSA HL S,-92324.00,25371173.67,COP,BC
3,2025-01-02 00:00:00,COMPRA INTL PAYPAL DISNEYPLU,-55423.01,25315750.66,COP,BC
4,2025-01-02 00:00:00,PAGO AUTOM TC MASTER PESOS,-35200.00,25280550.66,COP,BC
...,...,...,...,...,...,...
760,2025-10-13 09:29:59,Card charge (ESPRIT),-15.92,767.42,USD,PAYONEER
761,2025-10-13 09:29:59,Card charge (PATPRIMO 227 IBAGUE MU),-45.12,691.48,USD,PAYONEER
762,2025-10-13 09:29:59,Card charge (SURTIPLAZA ACQUA),-30.82,736.60,USD,PAYONEER
763,2025-11-10 15:16:41,Payment from Jala University,588.00,1226.37,USD,PAYONEER


In [8]:

    # Resumen en la moneda deseada
resumen = processing.summarize_dataframe(df, usd_to_cop=3700, currency='USD')

resumen

{'totalBalance': np.float64(4904.79),
 'entities': 3,
 'income': np.float64(27055.75),
 'expenses': np.float64(22150.96),
 'currency': 'USD'}

In [6]:
df

,MES,total_balance,income,expenses
0,2024-01,2194.99,1843.79,2382.04
1,2024-02,1283.38,2240.99,3154.84
2,2024-03,928.60,2129.73,2582.12
3,2024-04,2039.12,2677.34,2003.14
4,2024-05,1689.97,2129.52,2006.31
5,2024-06,1496.36,3100.73,3448.27
6,2024-07,1883.55,5691.82,1855.30
7,2024-08,1385.65,3597.24,2769.74
8,2024-09,21249.20,2594.68,2455.67
9,2024-10,21631.84,2706.90,1754.72


In [8]:
df = extract.extract_dataset(config.DATASET_ROOT_PATH, 2025)
df = extract.get_balanceMonthly(df, usd_to_cop=3700, currency='USD')
df

Subfolders found in C:\Users\USUARIO\Documents\Dev\finance\dataset\2025: ['Amerant', 'BC', 'Payoneer']
Extracting BC data...
Extracting data from folder: C:\Users\USUARIO\Documents\Dev\finance\dataset/2025/BC/
Processing file (<function extractExtractosFromFolderYearBC at 0x0000024AE14F6AC0>): Extracto_202503_Cuentas_de ahorro_T1.xlsx
Processing file (<function extractExtractosFromFolderYearBC at 0x0000024AE14F6AC0>): Extracto_202506_Cuentas_de ahorro_T2.xlsx
Processing file (<function extractExtractosFromFolderYearBC at 0x0000024AE14F6AC0>): Extracto_202509_Cuentas_de ahorro_T3.xlsx
BC data extracted for year C:\Users\USUARIO\Documents\Dev\finance\dataset 2025
Extracting Amerant data...
C:\Users\USUARIO\Documents\Dev\finance\dataset\2025\Amerant\INTLSAVINGS-7520 2025.xls
Amerant data extracted for year C:\Users\USUARIO\Documents\Dev\finance\dataset 2025
Extracting Payoneer data...
C:\Users\USUARIO\Documents\Dev\finance\dataset 2025
payoneer data extracted for year C:\Users\USUARIO\Doc

,MES,total_balance,income,expenses
0,2025-01,6948.14,1788.70,1832.55
1,2025-02,7482.66,2190.50,1631.89
2,2025-03,7456.63,2527.44,2314.91
3,2025-04,22709.50,3098.89,1801.64
4,2025-05,6457.27,2797.16,3691.44
5,2025-06,8041.94,5163.85,3517.19
6,2025-07,8238.19,3642.34,2145.64
7,2025-08,24022.63,2688.38,2986.95
8,2025-09,7662.79,1981.94,1720.81
9,2025-10,24022.73,0.20,507.73


In [5]:
pormes = processing.monthSummary(df)
pormes['balance_mes'] = pormes['ingreso_mes'] + pormes['gasto_mes']  # O restar ingreso_mes - abs(gasto_mes)
pormes

KeyError: 'FECHA'

In [ ]:
pormes['MES']

In [ ]:
df_2025['MONEDA']='COP'
df_2025

In [ ]:
print(pormes['MES'].dtype)

In [13]:
pormes['MES'] = pormes['MES'].dt.to_timestamp()
pormes = pormes.set_index('MES')

In [ ]:
pormes.index

In [ ]:
plt.figure(figsize=(10, 6))  # Tamaño del gráfico
# Graficar cada columna con un color diferente
plt.plot(pormes.index, pormes['saldo_inicio'], marker='o', label='Saldo Inicio', color='blue')
plt.plot(pormes.index, pormes['saldo_final'], marker='x', label='Saldo Final', color='red')
plt.plot(pormes.index, pormes['gasto_mes'].abs(), marker='o', label='Gasto', color='green')
plt.plot(pormes.index, pormes['ingreso_mes'], marker='o', label='Ingreso', color='yellow')
plt.plot(pormes.index, pormes['balance_mes'], marker='o', label='Balance', color='magenta')

# Añadir etiquetas y título
plt.xlabel('Meses')
plt.ylabel('Cantidad')
plt.title('Gráfico de Saldo, Gasto, Ingreso y Balance por Mes')
plt.xticks(rotation=45)  # Rotar los nombres de los meses para que sean más legibles
plt.legend(loc='best')  # Mostrar la leyenda

# Mostrar gráfico
plt.tight_layout()  # Asegurar que el diseño no se corte
plt.show()

In [14]:
json_data = extract.read_categories_class()
df_2025['CATEGORIA'] = df_2025['DESCRIPCION'].apply(processing.clasificar_transaccion, categorias=json_data)

In [ ]:
df_2025


In [ ]:
df_grouped = df_2025.groupby(['MES', 'CATEGORIA'])['CREDIT/DEBIT'].sum().reset_index()
df_grouped

In [ ]:
categories = df_grouped['CATEGORIA'].unique()
categories

In [ ]:
category_data = df_grouped[df_grouped['CATEGORIA'] == 'entretenimiento']
months = ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']
category_data = category_data.set_index('MES').reindex(months)
category_data['CREDIT/DEBIT'] = category_data['CREDIT/DEBIT'].fillna(0)
category_data['CATEGORIA'] = 'entretenimiento'
category_data

In [ ]:
# Group by 'MES' and 'CATEGORIA' and sum 'CREDIT/DEBIT'
df_grouped = df_2025.groupby(['MES', 'CATEGORIA'])['CREDIT/DEBIT'].sum().reset_index()

# Create a cyclical plot for each category
categories = df_grouped['CATEGORIA'].unique()
months = ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']

# Convert months to radians (to make the plot cyclical)
month_angles = np.linspace(0, 2 * np.pi, len(months), endpoint=False)

# Plot each category separately
for category in categories:
    category_data = df_grouped[df_grouped['CATEGORIA'] == category]
    #print(category_data)
    # Prepare the data for plotting
    category_data = category_data.set_index('MES').reindex(months)
    category_data['CREDIT/DEBIT'] = category_data['CREDIT/DEBIT'].fillna(0)
    category_data['CATEGORIA'] = category
    values = category_data['CREDIT/DEBIT'].apply(abs).values  # Get the absolute values of the credits/debits
    #print(values)
    # Set up polar plot
    fig, ax = plt.subplots(figsize=(10,8),subplot_kw={'projection': 'polar'})
    ax.set_title(f'Cyclic Plot by {category}')

    # Plot values for the category
    ax.plot(month_angles, values, marker='o', label=category)

    # Add grid and labels
    ax.fill(month_angles, values, alpha=0.3)
    ax.set_xticks(month_angles)
    ax.set_xticklabels(months)
    ax.set_rlabel_position(-22.5)
    
    plt.legend(loc="upper right")
    plt.show()
    


## Este por los meses de un año este caso 2025

In [24]:
file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia01Ene2026.xlsx")
df_oct_31dic = pd.read_excel(file)
df_oct_31dic = df_oct_31dic.drop(columns=['Referencia'])
df_oct_31dic = df_oct_31dic.rename(columns={'Fecha': 'FECHA', 'Descripción': 'DESCRIPCION', 'Valor': 'CREDIT/DEBIT'}) 

In [ ]:
df_oct_31dic['MONEDA']='COP'

In [ ]:
df_oct_31dic

In [ ]:
#Julio  2025
file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Jul2025_1.xlsx")
df_julio = pd.read_excel(file)
print(df_julio.shape)

file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Jul2025_2.xlsx")
df_julio = pd.concat([df_julio, pd.read_excel(file)], ignore_index=True)
print(df_julio.shape)

file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Jul2025_3.xlsx")
df_julio = pd.concat([df_julio, pd.read_excel(file)], ignore_index=True)
print(df_julio.shape)

file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Jul2025_4.xlsx")
df_julio =  pd.concat([df_julio, pd.read_excel(file)], ignore_index=True)
print(df_julio.shape)

file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Jul2025_5.xlsx")
df_julio =  pd.concat([df_julio, pd.read_excel(file)], ignore_index=True)

print(df_julio.shape)
df_julio

In [11]:
df_julio = df_julio.drop(columns=['Referencia'])
df_julio = df_julio.rename(columns={'Fecha': 'FECHA', 'Descripción': 'DESCRIPCION', 'Valor': 'CREDIT/DEBIT'})


In [ ]:
#Agosto  2025
file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Ago2025.xlsx")
df_ago = pd.read_excel(file)
print(df_ago.shape)

file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Ago2025 (1).xlsx")
df_ago = pd.concat([df_ago, pd.read_excel(file)], ignore_index=True)
print(df_ago.shape)

file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Ago2025 (2).xlsx")
df_ago = pd.concat([df_ago, pd.read_excel(file)], ignore_index=True)
print(df_ago.shape)

file = str(Path(config.DATASET_ROOT_PATH) / "BC/pormes/MovimientosTusCuentasBancolombia24Ago2025 (3).xlsx")
df_ago = pd.concat([df_ago, pd.read_excel(file)], ignore_index=True)
print(df_ago.shape)
df_ago

In [44]:
df_ago = df_ago.drop(columns=['Referencia'])
df_ago = df_ago.rename(columns={'Fecha': 'FECHA', 'Descripción': 'DESCRIPCION', 'Valor': 'CREDIT/DEBIT'})   

In [ ]:
df_ago

In [ ]:
df_2025.columns



In [ ]:
df_oct_22dic


In [ ]:
df_2025

In [ ]:
df_oct_31dic

In [34]:
# # Concatenate both DataFrames
df_combined = pd.concat([df_2025, df_oct_31dic], ignore_index=True)

In [ ]:
df_combined

In [ ]:

# # Concatenate both DataFrames
# df_combined = pd.concat([df_2024, df], ignore_index=True)

# # Convert FECHA to datetime format to ensure proper sorting
df_combined['FECHA'] = pd.to_datetime(df_combined['FECHA'], format='%Y/%m/%d')

# Sort by 'FECHA' in ascending order (default)
df_sorted = df_combined.sort_values(by='FECHA')
#df_sorted[['CREDIT/DEBIT', 'SALDO']] = df_sorted[['CREDIT/DEBIT', 'SALDO']].apply(lambda x: x.str.replace(',', '').astype(float))
df_sorted



In [ ]:
29755186.09

In [ ]:
df_sorted["SALDO"] = df_sorted["SALDO"].fillna(0)
df_sorted


In [ ]:


# Filtrar solo registros del mes de julio (mes=7)
df_sorted[(df_sorted["FECHA"].dt.month == 7) & (df_sorted["FECHA"].dt.year == 2025) & (df_sorted["FECHA"].dt.day == 1)]["SALDO"]


In [ ]:
df_sorted.loc[df_sorted['FECHA'] == '2025-01-07', 'SALDO']

In [ ]:
df_sorted.loc[df_sorted['FECHA'] == '2025-01-01', 'SALDO']

In [38]:
# # Paso 2: Ordenar el DataFrame por la columna FECHA
df_sorted = df_sorted.sort_values(by='FECHA').reset_index(drop=True)

# # Paso 3: Calcular nuevamente el SALDO a partir del primer saldo no cero
# # Encuentra el índice del primer saldo 0
zero_saldo_index = df_sorted.index[np.isclose(df_sorted['SALDO'], 0)].min()

# # Si hay algún saldo 0, tomar el saldo anterior y recalcular a partir de ahí
if zero_saldo_index > 0:
    saldo_inicial = df_sorted.loc[zero_saldo_index - 1, 'SALDO']
else:
    saldo_inicial = 0

# # Recalcular el saldo a partir de ese punto
df_sorted.loc[zero_saldo_index:, 'SALDO'] = saldo_inicial + df_sorted.loc[zero_saldo_index:, 'CREDIT/DEBIT'].cumsum()
df_sorted.to_csv('2024_ene.csv')

In [ ]:
df_sorted

In [ ]:
pormes = processing.monthSummary(df_sorted)
pormes['balance_mes'] = pormes['ingreso_mes'] + pormes['gasto_mes']  # O restar ingreso_mes - abs(gasto_mes)
pormes

In [41]:
json_data = extract.read_categories_class()
df_sorted['CATEGORIA'] = df_sorted['DESCRIPCION'].apply(processing.clasificar_transaccion, categorias=json_data)

In [ ]:
df_sorted

In [43]:
df_sorted.to_csv('2025_semI_oct_dic.csv', index=False)

In [40]:
# Definir el rango de fechas
fecha_inicio = "2025-05-01"
fecha_fin = "2025-08-16"

# Filtrar registros entre esas fechas
df_rango = df_sorted[
    df_sorted["FECHA"].between(fecha_inicio, fecha_fin)
]

df_rango.to_csv('2025_33.csv', index=False)


In [ ]:
# Crear copia segura del rango
df_rango = df_rango.copy()

# Convertir a numérico sin warnings
df_rango.loc[:, "CREDIT/DEBIT"] = pd.to_numeric(df_rango["CREDIT/DEBIT"], errors="coerce")

# Calcular totales
total_credit = df_rango.loc[df_rango["CREDIT/DEBIT"] > 0, "CREDIT/DEBIT"].sum()
total_debit = df_rango.loc[df_rango["CREDIT/DEBIT"] < 0, "CREDIT/DEBIT"].sum()

print("Total Créditos:", total_credit)
print("Total Débitos:", total_debit)


In [ ]:
pormes = processing.monthSummary(df_sorted)
pormes['balance_mes'] = pormes['ingreso_mes'] + pormes['gasto_mes']  # O restar ingreso_mes - abs(gasto_mes)
pormes

In [ ]:
pormes['balance_mes'].sum()

In [ ]:
pormes = processing.monthSummary(df_sorted)
pormes['balance_mes'] = pormes['ingreso_mes'] + pormes['gasto_mes']  # O restar ingreso_mes - abs(gasto_mes)
pormes

In [ ]:
plt.figure(figsize=(10, 6))  # Tamaño del gráfico
# Graficar cada columna con un color diferente
plt.plot(pormes.index, pormes['saldo_inicio'], marker='o', label='Saldo Inicio', color='blue')
plt.plot(pormes.index, pormes['saldo_final'], marker='x', label='Saldo Final', color='red')
plt.plot(pormes.index, pormes['gasto_mes'].abs(), marker='o', label='Gasto', color='green')
plt.plot(pormes.index, pormes['ingreso_mes'], marker='o', label='Ingreso', color='yellow')
plt.plot(pormes.index, pormes['balance_mes'], marker='o', label='Balance', color='magenta')

# Añadir etiquetas y título
plt.xlabel('Meses')
plt.ylabel('Cantidad')
plt.title('Gráfico de Saldo, Gasto, Ingreso y Balance por Mes')
plt.xticks(rotation=45)  # Rotar los nombres de los meses para que sean más legibles
plt.legend(loc='best')  # Mostrar la leyenda

# Mostrar gráfico
plt.tight_layout()  # Asegurar que el diseño no se corte
plt.show()

In [ ]:
pormes['balance_mes'] = pormes_ene['ingreso_mes'] + pormes_ene['gasto_mes']  # O restar ingreso_mes - abs(gasto_mes)
pormes

In [ ]:
df_2025.columns

In [ ]:
df_2025 = processing.monthSummary(df_sorted)
df_2025

In [ ]:
df_sorted.columns


In [ ]:
df = pd.concat([df_2024, df_sorted], ignore_index=True)
df

In [ ]:
# Group by 'MES' and 'CATEGORIA' and sum 'CREDIT/DEBIT'
df_grouped = df.groupby(['MES', 'CATEGORIA'])['CREDIT/DEBIT'].sum().reset_index()

# Create a cyclical plot for each category
categories = df_grouped['CATEGORIA'].unique()
months = ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']

# Convert months to radians (to make the plot cyclical)
month_angles = np.linspace(0, 2 * np.pi, len(months), endpoint=False)

# Plot each category separately
for category in categories:
    category_data = df_grouped[df_grouped['CATEGORIA'] == category]
    #print(category_data)
    # Prepare the data for plotting
    category_data = category_data.set_index('MES').reindex(months)
    category_data['CREDIT/DEBIT'] = category_data['CREDIT/DEBIT'].fillna(0)
    category_data['CATEGORIA'] = category
    values = category_data['CREDIT/DEBIT'].apply(abs).values  # Get the absolute values of the credits/debits
    #print(values)
    # Set up polar plot
    fig, ax = plt.subplots(figsize=(10,8),subplot_kw={'projection': 'polar'})
    ax.set_title(f'Cyclic Plot by {category}')

    # Plot values for the category
    ax.plot(month_angles, values, marker='o', label=category)

    # Add grid and labels
    ax.fill(month_angles, values, alpha=0.3)
    ax.set_xticks(month_angles)
    ax.set_xticklabels(months)
    ax.set_rlabel_position(-22.5)
    
    plt.legend(loc="upper right")
    plt.show()
    


In [80]:
# Asegúrate de que tienes una columna de fecha en tu DataFrame. Si es de tipo string, conviértela en formato de fecha.
df_sorted['FECHA'] = pd.to_datetime(df_sorted['FECHA'])  # Ajusta el nombre de la columna 'FECHA' si es diferente

# Extraer el mes de la columna de fechas
#df_sorted['MES'] = df_sorted['FECHA'].dt.strftime('%m')  # Puedes usar '%m' para obtener el número del mes o '%B' para el nombre

# Crear una nueva columna 'CATEGORIA' aplicando la función de clasificación
df_sorted['CATEGORIA'] = df_sorted['DESCRIPCION'].apply(processing.clasificar_transaccion, categorias=json_data)

# Agrupar por 'CATEGORIA' y 'MES', sumando los valores de 'CREDIT/DEBIT'
df_classified = df_sorted.groupby(['CATEGORIA', 'MES'])['CREDIT/DEBIT'].sum().reset_index()

# Ordenar los resultados por 'CREDIT/DEBIT' en orden descendente
df_classified = df_classified.sort_values(by='CREDIT/DEBIT', ascending=False)


In [82]:
df_sorted.to_csv('2025.csv')

In [ ]:
df_classified = df_classified.sort_values(by=['MES', 'CREDIT/DEBIT'], ascending=[True, False])
df_classified

In [ ]:
total_outcome_mes02 = df_classified[(df_classified['CREDIT/DEBIT'] < 0) & (df_classified['MES']=='2025-02')]['CREDIT/DEBIT'].sum()
print(total_outcome_mes02)

In [ ]:
df_classified

In [ ]:
df_sorted[(df_sorted['CATEGORIA']=='intereses') & (df_sorted['MES']=='2025-02')]


In [ ]:
total_outcome_mes02 = df_classified[(df_classified['CREDIT/DEBIT'] < 0) & (df_classified['MES']=='2025-02')]['CREDIT/DEBIT'].sum()
print(total_outcome_mes02)

In [ ]:
df_sorted[(df_sorted['CATEGORIA']=='transferencia') & (df_sorted['DESCRIPCION'] == 'OTROS')]['CREDIT/DEBIT'].sum()

In [ ]:
# Group by 'DESCRIPCION' and sum the 'CREDIT/DEBIT' values
gasto_por_descripcion = df_sorted.groupby('DESCRIPCION')['CREDIT/DEBIT'].sum().reset_index()

# Filter out positive values to only keep the expenses
gasto_por_descripcion = gasto_por_descripcion[gasto_por_descripcion['CREDIT/DEBIT'] < 0]

# Rename the column for clarity
gasto_por_descripcion = gasto_por_descripcion.rename(columns={'CREDIT/DEBIT': 'TOTAL_GASTO'})

# Display the result
gasto_por_descripcion

In [ ]:


# Función para clasificar las transacciones
def clasificar_transaccion(descripcion, categorias):
    for categoria, detalles in categorias.items():
        for keyword in detalles["lista"]:
            if keyword in descripcion:
                return categoria
    return 'otros'  # Si no coincide con ninguna categoría

# Crear una nueva columna 'CATEGORIA' en el DataFrame
df['CATEGORIA'] = df['DESCRIPCION'].apply(clasificar_transaccion, categorias=json_data)

# Mostrar el DataFrame clasificado
print(df)

In [ ]:
df.info()

In [ ]:
df_sorted

In [91]:
df_sorted.to_csv('2025.csv')

In [ ]:
# Group by 'DESCRIPCION' and sum the 'CREDIT/DEBIT' values
gasto_por_descripcion = df_sorted.groupby(['DESCRIPCION', 'MES'])['CREDIT/DEBIT'].sum().reset_index()

# Filter out positive values to only keep the expenses
gasto_por_descripcion = gasto_por_descripcion[gasto_por_descripcion['CREDIT/DEBIT'] < 0]

# Rename the column for clarity
gasto_por_descripcion = gasto_por_descripcion.rename(columns={'CREDIT/DEBIT': 'TOTAL_GASTO'})

# Display the result
gasto_por_descripcion

In [ ]:

# Excluir los meses de diciembre y junio
result_excluding_december = result[(result['MES'] != '2024-12') & (result['MES'] != '2024-06')]

# Encontrar el mes con el mínimo gasto
max_gasto_mes_excluding_december = result_excluding_december.loc[result_excluding_december['gasto_mes'].idxmin()]
max_gasto_mes_excluding_december


In [ ]:
# Exclude the months of December and June
result_excluding_december_june = result.query("MES != '2024-12' & MES != '2024-06'")

# Display the result
result_excluding_december_june

In [ ]:
# Paso 4: Crear una columna 'balance_mes' con la diferencia entre ingresos y gastos
result['balance_mes'] = result['ingreso_mes'] + result['gasto_mes']  # O restar ingreso_mes - abs(gasto_mes)

# Mostrar el resultado
result.to_csv('resumenpormes.csv')
result

In [ ]:
result_sorted_by_gasto_mes = result.sort_values(by='gasto_mes')

# Mostrar el resultado
result_sorted_by_gasto_mes

In [10]:


# Paso 3: Agrupar por 'MES' y 'DESCRIPCION' y calcular el gasto/ingreso por descripción
result2 = df_sorted.groupby(['DESCRIPCION']).agg(
    GASTO=('CREDIT/DEBIT', 'sum'),  # Sumar los valores de CREDIT/DEBIT para cada descripción,   # Sumar los valores de gasto
    VECES=('CREDIT/DEBIT', 'count')          
).reset_index()


# Mostrar el resultado
result2.to_csv("2024_descripcion.csv")

In [ ]:
df_sorted.columns

In [ ]:
# fecha en formato YYYY-MM a la fecha actual 
formato_fecha_anterior = (datetime.now() - relativedelta(months=1)).strftime('%Y-%m')

print(formato_fecha_anterior)


In [101]:
def gastoMes(df, anio_mes='2024-12'):
    df_diciembre = df_sorted[df_sorted['MES'] == anio_mes]

    # Paso 4: Agrupar por 'MES' y 'DESCRIPCION' y calcular el gasto/ingreso por descripción
    result_diciembre = df_diciembre.groupby(['MES', 'DESCRIPCION']).agg(
        GASTO=('CREDIT/DEBIT', 'sum'),  # Sumar los valores de CREDIT/DEBIT para cada descripción,   # Sumar los valores de gasto
        VECES=('CREDIT/DEBIT', 'count')          
    ).reset_index()

    # Ordenar el resultado por MES para mejor visualización
    result_diciembre = result_diciembre.sort_values(by='MES')

    # Mostrar el resultado
    return result_diciembre


In [102]:
df_sorted['MES'] = df_sorted['MES'].astype(str)

In [ ]:
gastoMes(df_sorted, '2025-11')

In [ ]:
gastoMes(df_sorted, formato_fecha_anterior)

In [186]:
df_2025 = df_sorted

## AMERANT account

In [4]:
df_amerant = extract.extract_amerant(config.DATASET_ROOT_PATH,2025)


,FECHA,DESCRIPCION,CREDIT/DEBIT,MONEDA,ENTIDAD
0,2025-01-16,SERVICE CHARGE INCOMING WIRE FEE,-12.00,USD,AMERANT
1,2025-01-31,FED TAX WITHHELD,-0.04,USD,AMERANT
2,2025-01-31,INTEREST CREDIT,0.18,USD,AMERANT
3,2025-02-12,CREDIT MEMO FEE REIMBURSEMENT,1.65,USD,AMERANT
4,2025-02-28,FED TAX WITHHELD,-0.04,USD,AMERANT
5,2025-02-28,INTEREST CREDIT,0.17,USD,AMERANT
6,2025-03-12,WIRE IN-DOM c0a895e7-3467-483c -992b-502d4091b...,211.50,USD,AMERANT
7,2025-03-12,SERVICE CHARGE INCOMING WIRE FEE,-12.00,USD,AMERANT
8,2025-03-31,FED TAX WITHHELD,-0.04,USD,AMERANT
9,2025-03-31,INTEREST CREDIT,0.18,USD,AMERANT


In [157]:
df_amerant_2023 = read_amerant(config.DATASET_ROOT_PATH, 2023)
df_amerant_2023.to_csv('2023_amerant.csv')
df_amerant_2024 = read_amerant(config.DATASET_ROOT_PATH, 2024)
df_amerant_2024.to_csv('2024_amerant.csv')
df_amerant_2025 = read_amerant(config.DATASET_ROOT_PATH, 2025)
df_amerant_2025.to_csv('2025_amerant.csv')

In [ ]:
df_amerant_2025.columns

In [ ]:
df_diciembre = df_diciembre.reindex(columns=['FECHA', 'DESCRIPCION', 'CREDIT/DEBIT', 'SALDO', 'MES'])
df_diciembre

In [159]:
# Paso 3: Agrupar por 'MES' y 'DESCRIPCION' y calcular el gasto/ingreso por descripción
result2 = df_amerant_2025.groupby(['DESCRIPCION']).agg(
    GASTO=('CREDIT/DEBIT', 'sum'),  # Sumar los valores de CREDIT/DEBIT para cada descripción,   # Sumar los valores de gasto
    VECES=('CREDIT/DEBIT', 'count')          
).reset_index()


# Mostrar el resultado
result2.to_csv("2025_amerant_descripcion.csv")

In [ ]:
df_amerant_2025

In [ ]:
# Paso 1: Asegurarse de que la columna FECHA sea de tipo datetime
df_amerant_2025['FECHA'] = pd.to_datetime(df_amerant_2025['FECHA'])

# Paso 2: Crear una columna 'MES' que contenga año y mes
df_amerant_2025['MES'] = df_amerant_2025['FECHA'].dt.to_period('M')  # Agrupa por mes y año

# Paso 3: Agrupar por 'MES' y calcular los valores solicitados
result = df_amerant_2025.groupby('MES').agg(
    saldo_inicio=('SALDO', 'first'),          # Saldo del inicio del mes
    saldo_final=('SALDO', 'last'),            # Saldo del final del mes
    gasto_mes=('CREDIT/DEBIT', lambda x: x[x < 0].sum()),   # Gasto por mes (sumar valores negativos)
    ingreso_mes=('CREDIT/DEBIT', lambda x: x[x > 0].sum())  # Ingreso por mes (sumar valores positivos)
).reset_index()

# Mostrar el resultado
result

In [ ]:
# Paso 4: Crear una columna 'balance_mes' con la diferencia entre ingresos y gastos
result['balance_mes'] = result['ingreso_mes'] + result['gasto_mes']  # O restar ingreso_mes - abs(gasto_mes)

# Mostrar el resultado
result

## PAYONEER

In [152]:
# Load the Excel file, skipping the first line and using the second line as the header
def read_payoneer(folder, year):
    print(folder, year)
    file = str(Path(folder) / f"Payoneer/{year}/USD transactions 01 Jan 2025-22 Dec 2025.csv")
    print( file)    
    df = pd.read_csv(file)
    df['FECHA'] = pd.to_datetime(
        df['Transaction date'] + ' ' + df['Transaction time']
        )
    df['CREDIT/DEBIT'] = (
        df['Credit amount'].fillna(0) +
        df['Debit amount'].fillna(0)
        )
    df = df.rename(columns={
            'Description': 'DESCRIPCION',
            'Running balance': 'SALDO',
            'Currency': 'MONEDA'
        })
    
    #df = df.drop(df.index[-1])
    #df = df.sort_values(by='Date').reset_index(drop=True)
    #df['CREDIT/DEBIT'] = df.apply(
    #    lambda row: -row['Debit Amount'] if pd.notnull(row['Debit Amount']) else row['Credit Amount'], axis=1
    #)
    #df = df.drop(columns=['Debit Amount', 'Credit Amount'])
    #df = df.rename(columns={'Description': 'DESCRIPCION', 'Date': 'FECHA', 'Running Balance': 'SALDO'})
    #df = df.drop(columns=['Check Number'])
    return df

In [4]:
df = extract.extract_payoneer(config.DATASET_ROOT_PATH, 2025)

AttributeError: module 'app.core.extract' has no attribute 'extract_payoneer'

In [ ]:
df_payoneer = read_payoneer(config.DATASET_ROOT_PATH, 2025)
df_payoneer.columns

In [154]:
df_payoneer = df_payoneer[['FECHA', 'DESCRIPCION', 'SALDO', 'CREDIT/DEBIT', 'MONEDA']]

In [ ]:
df_payoneer

In [162]:
df_payoneer = df_payoneer.sort_values('FECHA').reset_index(drop=True)

In [ ]:
# Paso 1: Asegurarse de que la columna FECHA sea de tipo datetime
df_payoneer['FECHA'] = pd.to_datetime(df_payoneer['FECHA'])

# Paso 2: Crear una columna 'MES' que contenga año y mes
df_payoneer['MES'] = df_payoneer['FECHA'].dt.to_period('M')  # Agrupa por mes y año

# Paso 3: Agrupar por 'MES' y calcular los valores solicitados
result = df_payoneer.groupby('MES').agg(
    saldo_inicio=('SALDO', 'first'),          # Saldo del inicio del mes
    saldo_final=('SALDO', 'last'),            # Saldo del final del mes
    gasto_mes=('CREDIT/DEBIT', lambda x: x[x < 0].sum()),   # Gasto por mes (sumar valores negativos)
    ingreso_mes=('CREDIT/DEBIT', lambda x: x[x > 0].sum())  # Ingreso por mes (sumar valores positivos)
).reset_index()

# Mostrar el resultado
result

In [ ]:
df_payoneer

In [ ]:
import yfinance as yf

pair = yf.Ticker("COPUSD=X")
data = pair.history(period="1d")
rate_today = data["Close"][-1]

print("Tipo de cambio COP→USD:", rate_today)

In [ ]:
df_amerant_2025.columns

In [191]:
df_2025['ENTIDAD'] = "BC"
df_amerant_2025['ENTIDAD'] = "Amerant"
df_payoneer['ENTIDAD'] = "Payoneer"

In [ ]:
print(f"BC \n{df_2025.columns}")
print(f"Amerant \n{df_amerant_2025.columns}")
print(f"Payoneer \n{df_payoneer.columns}")

In [196]:
df_2025['MONEDA'] = 'COP'

In [ ]:
df_2025

In [201]:
for df in [df_2025, df_amerant_2025, df_payoneer]:
    df['FECHA'] = pd.to_datetime(df['FECHA']).dt.date

In [202]:
df_all = pd.concat([df_2025, df_amerant_2025, df_payoneer], ignore_index=True)

In [ ]:
df_all

In [204]:
saldo_actual = (
    df_all.sort_values('FECHA')
      .groupby('ENTIDAD')
      .tail(1)
      .copy()
)

In [ ]:
saldo_actual

In [208]:
saldo_por_entidad = (
    df_all.groupby('ENTIDAD', as_index=False)
      .tail(1)
      .loc[:, ['ENTIDAD', 'SALDO', 'MONEDA']]
)

In [ ]:
saldo_por_entidad

In [211]:
saldo_por_entidad['SALDO_USD_HOY'] = np.where(
    saldo_por_entidad['MONEDA'] == 'COP',
    saldo_por_entidad['SALDO'] * rate_today,
    saldo_por_entidad['SALDO']
)

In [ ]:
saldo_por_entidad

In [ ]:
saldo_por_entidad['SALDO_USD_HOY'].sum()